In [1]:
import pandas as pd

ANOMALOUS_CLUSTERS = [3, 1]

cluster_file = "Clustering/clustering_real_2026_03_19_sent_volvo/clustering.csv"
original_file = "Dataset/mergedTimeStampDiff.csv"
original_file_labeled = "Dataset/mergedTimeStampDiffLabeled.csv"

anomalous_points_file = "Dataset/anomalousPoints.csv"
anomalous_points_file_labeled = "Dataset/anomalousPointsLabeled.csv"

volvo_short_dataset_file = "Dataset/Processed Dataset/volvoShort.csv"

volvo_train_file = "Dataset/Processed Dataset/volvoTrain.csv"
volvo_test_file = "Dataset/Processed Dataset/volvoTest.csv"

In [2]:
# Check that mergeTimeStampDiff first row is gone\n
# Use on correct clustering file\n
# Check correct anomaly clusters\n
# Check 10 extra anomalies (duplicates)\n

cluster_df = pd.read_csv(cluster_file)
# Load original data
original_df = pd.read_csv(original_file)

anomalous_windows = cluster_df[
    cluster_df['cluster'].isin(ANOMALOUS_CLUSTERS)
]
anomalous_windows.head()

print("Cluster counts: ")
print(anomalous_windows['cluster'].value_counts())

Cluster counts: 
cluster
3    440
1    163
Name: count, dtype: int64


## Get a list of all points (call them anomalous points) in anomalous windows

In [3]:
# Create a boolean mask aligned with original_df, initialized to False (1 entry in mask is 1 row in original_df)
mask = pd.Series(False, index=original_df.index)

timestamps = original_df['timestamp'].values
starts = anomalous_windows['start_timestamp'].values
ends = anomalous_windows['end_timestamp'].values

# zip puts [(starts1, ends1), (starts2, ends2), ...]
# searchsorted assumes sorted array (timestamps are sorted from the start)
for start, end in zip(starts, ends):
    start_idx = timestamps.searchsorted(start, side='left')
    end_idx = timestamps.searchsorted(end, side='right')
    mask.iloc[start_idx:end_idx] = True

anomalous_df = original_df.loc[mask, ['timestamp']]
anomalous_df.to_csv(anomalous_points_file, index=False)

## For each anomalous point, check all windows it belongs to and assign an anomaly label if >=50% windows are anomalous

In [4]:
anomalous_df = pd.read_csv(anomalous_points_file)
cluster_df = pd.read_csv(cluster_file)

# Label clusters
cluster_df['is_anomalous_cluster'] = cluster_df['cluster'].isin(ANOMALOUS_CLUSTERS).astype(int)

# Initialize labels
anomalous_df['anomaly_label'] = 0

starts = cluster_df['start_timestamp'].values
ends = cluster_df['end_timestamp'].values
labels = cluster_df['is_anomalous_cluster'].values
timestamps = anomalous_df['timestamp'].values

for idx, timestamp in enumerate(timestamps):

    # If starts = [1, 6, 11], ends = [51, 56, 61], timestamp = 7
    # mask = [1 <= 7, 6 <= 7, 11 <= 7] & [51 >= 7, 56 >= 7, 61 >= 7]
    # mask = [true, true, false]
    mask = (starts <= timestamp) & (ends >= timestamp)

    overlap_labels = labels[mask]

    # if true set current timestamp anomaly_label column to 1
    # iloc[row, column] = 1
    if overlap_labels.mean() >= 0.5:
        anomalous_df.iloc[idx, anomalous_df.columns.get_loc('anomaly_label')] = 1
    print(f"\r{idx}/{len(timestamps)} processed", end='', flush=True)

# Save result
anomalous_df.to_csv(anomalous_points_file_labeled, index=False)

5805/5806 processed

## Merge the labels into original file

In [5]:
import pandas as pd
anomalous_df = pd.read_csv(anomalous_points_file_labeled)
original_df = pd.read_csv(original_file)

print("anomalies counts in anomaly_label: ")
print(anomalous_df['anomaly_label'].value_counts())

# merge original_df with anomalous_df: original_df gets the values of anomaly label where timestamps match, NAN otherwise
original_df = original_df.merge(
    anomalous_df[['timestamp', 'anomaly_label']],
    on='timestamp',
    how='left'
)

# New column label gets values of anomaly_label if it exists, otherwise 0
original_df['label'] = original_df['anomaly_label'].fillna(0).astype(int)

# Drop useless anomaly_label column (replaced by label above)
original_df.drop(columns=['anomaly_label'], inplace=True)
original_df.to_csv(original_file_labeled, index=False)

anomalies counts in anomaly_label: 
anomaly_label
1    3304
0    2502
Name: count, dtype: int64


In [6]:
# Some events have same timestamp but different duration (around 10 ts), which mean they duplicate on the merge (since we merge on timestamp) ^
import pandas as pd
labeled_set_df = pd.read_csv(original_file_labeled)

duplicates_mask = labeled_set_df.duplicated(subset=["timestamp", "duration", "time_since_last_timestamp"], keep="first")
non_duplicate_mask = ~duplicates_mask

#duplicates_df = labeled_set_df[duplicates_mask] # Just for error checking
deduplicated_df = labeled_set_df[non_duplicate_mask]

#duplicates_df.to_csv("Dataset/duplicates.csv") # Just for error checking
deduplicated_df.to_csv(original_file_labeled, index=False)

print(non_duplicate_mask.value_counts())

True     19390041
False          86
Name: count, dtype: int64


In [7]:
import pandas as pd
anomalous_points_df = pd.read_csv(anomalous_points_file_labeled)
labeled_set_df = pd.read_csv(original_file_labeled)
original_df = pd.read_csv(original_file)

print("points in original dataset: ")
print(original_df.shape)
print("anomalies counts in anomalousPointsLabeled: ")
print(anomalous_points_df['anomaly_label'].value_counts())
print("\n anomalies counts in mergedTimeStampDiffLabeled: ")
print(labeled_set_df['label'].value_counts())
print()

anomaliesInPointsSet = anomalous_points_df[anomalous_points_df['anomaly_label'] == 1]
anomaliesInLabeledSet = labeled_set_df[labeled_set_df['label'] == 1]

outliers = labeled_set_df[
    ~labeled_set_df['timestamp'].isin(
        anomalous_points_df.loc[anomalous_points_df['anomaly_label'] == 1, 'timestamp']
    )
]
print(outliers[outliers['label'] == 1])
# How many of each label there are, should all be 1
labeled_set_df[labeled_set_df['label'] == 1]['timestamp'].value_counts().head(10)

timestamp
2025-11-17 05:24:57.428154+00:00    4
2025-11-17 09:40:59.691033+00:00    2
2025-11-17 08:52:46.351453+00:00    2
2025-11-17 06:31:01.319895+00:00    2
2025-11-17 07:36:45.935539+00:00    2
2025-11-17 09:16:31.882762+00:00    2
2025-11-17 06:13:37.854673+00:00    2
2025-11-17 06:44:02.404319+00:00    2
2025-11-17 06:15:01.686185+00:00    2
2025-11-17 04:45:03.825782+00:00    2
Name: count, dtype: int64

# Creating the test/train sets

In [16]:
import pandas as pd

start_date = "2025-11-26"
end_date = "2025-12-14" # not inclusive

df = pd.read_csv(original_file_labeled)
df = df.drop(columns=['metric'])
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, format='ISO8601')

df = df[(df['timestamp'] >= start_date) & (df['timestamp'] <= end_date)]

print("Start timestamp: \n", df.iloc[0])
print()
print("End timestamp: \n", df.iloc[-1])

df.to_csv(volvo_short_dataset_file, index=False)

Start timestamp: 
 timestamp                    2025-11-26 00:00:06.255619+00:00
duration                                             0.006405
time_since_last_timestamp                            12.01882
label                                                       0
Name: 8743432, dtype: object

End timestamp: 
 timestamp                    2025-12-13 23:59:59.719537+00:00
duration                                             0.005695
time_since_last_timestamp                            0.006008
label                                                       0
Name: 15650205, dtype: object


In [20]:
import pandas as pd

df = pd.read_csv(volvo_short_dataset_file)

trainset_percentage = 0.8

split = int(len(df) * trainset_percentage)

df_train = df.iloc[:split]
df_test = df.iloc[split:]

print(len(df_train), len(df_test))
print("df_train last timestamp:", df_train["timestamp"].iloc[-1])
print("df_test first timestamp:", df_test["timestamp"].iloc[0])

5525419 1381355
df_train last timestamp: 2025-12-10 12:43:09.072558+00:00
df_test first timestamp: 2025-12-10 12:43:09.197584+00:00


In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
feature_columns = ['duration', 'time_since_last_timestamp']

print("Training set columns before scaling: ", df_train)
df_train.loc[:, feature_columns] = scaler.fit_transform(df_train[feature_columns])
print("Training set columns after scaling: ", df_train)

print("Test set columns before scaling: ", df_test)
df_test.loc[:, feature_columns] = scaler.transform(df_test[feature_columns])
print("Test set columns after scaling: ", df_test)

df_train.to_csv(volvo_train_file, index=False)
df_test.to_csv(volvo_test_file, index=False)

Training set columns after scaling:                                  timestamp  duration  \
0        2025-11-26 00:00:06.255619+00:00 -0.004069   
1        2025-11-26 00:00:06.834617+00:00 -0.062109   
2        2025-11-26 00:00:06.840622+00:00 -0.044793   
3        2025-11-26 00:00:06.846600+00:00 -0.142067   
4        2025-11-26 00:00:06.850609+00:00 -0.214447   
...                                   ...       ...   
5525414  2025-12-10 12:43:08.615611+00:00 -0.034654   
5525415  2025-12-10 12:43:08.665612+00:00 -0.071992   
5525416  2025-12-10 12:43:09.050583+00:00  0.225642   
5525417  2025-12-10 12:43:09.050583+00:00  0.200219   
5525418  2025-12-10 12:43:09.072558+00:00  0.001892   

         time_since_last_timestamp  label  
0                        14.380236      0  
1                         0.429024      0  
2                        -0.269758      0  
3                        -0.269791      0  
4                        -0.272193      0  
...                            ...    

In [22]:
df_train['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, format='ISO8601')
df_test['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, format='ISO8601')
df_train['date'] = df_train['timestamp'].dt.date
df_test['date'] = df_test['timestamp'].dt.date
print(df_train.groupby('date')['label'].value_counts())
print(df_test.groupby('date')['label'].value_counts())

date        label
2025-11-26  0         90593
2025-11-27  0        524465
2025-11-28  0        518038
2025-11-29  0         85329
2025-11-30  0         83791
            1           165
2025-12-01  0        543365
2025-12-02  0        529817
2025-12-03  0        554874
2025-12-04  0        527245
2025-12-05  0        546014
2025-12-06  0         86223
2025-12-07  0         84990
2025-12-08  0        548303
2025-12-09  0        528422
2025-12-10  0        273755
            1            30
Name: count, dtype: int64
date        label
2025-12-10  0        263485
2025-12-11  0        553112
2025-12-12  0        467176
            1           109
2025-12-13  0         97418
            1            55
Name: count, dtype: int64
